# 🗂️ 01 · Data Sources & Cleaning

> **Chapter 1.** How 195,098 missense variants were assembled from
> three public resources.

## The three primary sources

| Source | Version | What it gives | Build |
|---|---|---|---|
| **ClinVar** | 2024-01 variant_summary.txt.gz | Pathogenic / benign labels | GRCh38 |
| **gnomAD** | v2.1.1 exomes + v4.1.0 genomes | Allele frequency, AN, AC | GRCh38 |
| **dbNSFP** | 5.3.1a | Conservation, AA properties, domains, gene constraint | GRCh38* |

\* `dbNSFP5.3.1a_grch37.gz` is the filename; its `pos(1-based)`
column is in fact GRCh38 (the dbNSFP 5.x convention flipped). See the
"Build correction" entry in `docs/CHANGELOG.md` for the full audit.

---

In [ ]:
from __future__ import annotations
import sys
from pathlib import Path as _Path
sys.path.insert(0, str(_Path.cwd().parent))
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False})

REPO = Path.cwd().parent
train = pd.read_parquet(REPO / "data/splits/train.parquet")
val = pd.read_parquet(REPO / "data/splits/val.parquet")
test = pd.read_parquet(REPO / "data/splits/test.parquet")

total = len(train) + len(val) + len(test)
print(f"Total labeled missense variants: {total:,}")
print(f"  train  {len(train):>7,} ({len(train)/total:.1%})")
print(f"  val    {len(val):>7,} ({len(val)/total:.1%})")
print(f"  test   {len(test):>7,} ({len(test)/total:.1%})")

## Label balance across splits

A good split preserves the pathogenic / benign ratio so no model gets
a trivial "predict-the-majority" shortcut.

In [ ]:
rates = {
    "train": train["label"].mean(),
    "val": val["label"].mean(),
    "test": test["label"].mean(),
}
for k, v in rates.items():
    print(f"  {k}: pathogenic rate = {v:.4f}  ({int((v*({'train':len(train),'val':len(val),'test':len(test)}[k]))):,} pathogenic)")

gap = max(rates.values()) - min(rates.values())
print(f"\nMax rate gap across splits: {gap:.4f}  (contract: < 0.08)")

## Variant-identity schema

Every row is keyed by a canonical variant string. This is the spine
that external datasets (denovo-db, ProteinGym) use to join against
our cache.

In [ ]:
print("Example rows (first 5 of train):")
print(train[["variant_key", "chr", "pos", "ref", "alt", "gene", "ref_aa", "alt_aa", "label"]].head().to_string(index=False))

### The missense filter — the single most important cleaning step

In the pre-audit baseline, 64% of *pathogenic* rows had null
`ref_aa` OR `alt_aa`. Those were stop-gained / splice-site / UTR
variants leaking into a "missense" dataset. The model was partly
learning *"if AA features are null, it's pathogenic"* — pure leakage.

The fix: drop every row where `ref_aa` or `alt_aa` is null. See
`src/feature_analysis.py::filter_missense_only`. All 195,098 rows
in the current splits pass this filter (asserted in
`tests/test_verify_no_leakage.py`).

In [ ]:
# Confirm the filter holds on the committed splits.
for name, df in [("train", train), ("val", val), ("test", test)]:
    bad = int((df["ref_aa"].isna() | df["alt_aa"].isna()).sum())
    assert bad == 0, f"{name}: {bad} non-missense rows leaked in!"
print("✅ all three splits pass the missense-only filter (zero non-missense rows)")

## Chromosomal distribution

Variants per chromosome. This is a sanity check — a systematic
over-representation of, say, chrX would suggest a capture-bias
problem.

In [ ]:
chr_counts = pd.concat([train, val, test])["chr"].value_counts().sort_index(
    key=lambda s: pd.to_numeric(s.str.replace("X", "23").str.replace("Y", "24").str.replace("MT", "25"), errors="coerce")
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(chr_counts.index.astype(str), chr_counts.values, color="#2c3e50")
ax.set_xlabel("Chromosome")
ax.set_ylabel("Variant count")
ax.set_title("Variant distribution across chromosomes (train + val + test)",
             fontweight="bold", loc="left")
fig.tight_layout()
plt.show()

## Reproduce

```bash
# End-to-end data assembly (slow — hours, needs dbNSFP + gnomAD raw):
python -m src.clinvar_cleaning
python -m src.gnomad_extraction
python -m src.dbnsfp_extraction
python -m src.data_merge
python -m src.feature_analysis      # missense filter
python -m src.data_splitting        # paralog-aware family split
python -m src.gnomad_constraint     # Phase 2 step 1 features
```

See `docs/CHANGELOG.md` "0.2.0 — Honest Baseline" for the original
cleaning decisions.